# CapLife AI — Colab quickstart

**Goal:** Test the trained GRU engine from the same codebase as the website. No retraining required. This is an **unvalidated, synthetic-only research demonstrator**, not a reliable AEC lifetime predictor.

**Before starting:** Download `CapLife_AI_Starter.zip` to your computer.

## 1. Upload your project ZIP
Run the next cell and choose **CapLife_AI_Starter.zip**. This is a new, separate project from your original Random Forest notebook.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
filename = next(iter(uploaded))
assert filename.endswith(".zip"), "Please upload the CapLife_AI_Starter.zip file."
with zipfile.ZipFile(filename) as z:
    z.extractall("/content")
print("Project extracted to /content/CapLife_AI_Starter")


## 2. Load the trained AI
This will import our GRU and the preprocessing instructions. Do **not** click Run all in the old RUL notebook.

In [ ]:
import os
os.chdir("/content/CapLife_AI_Starter")

from caplife.predict import load_engine, assess_capacitor
from caplife.data import validate_measurements
import pandas as pd

model, metadata = load_engine("artifacts")
print("Loaded:", metadata["model_id"])
print("Network:", metadata["model_type"])
print("Needs", metadata["sequence_length"], "historical measurements per prediction.")


## 3. Make a demonstration RUL prediction
We select a simulated capacitor, but keep only the **first 15 measurements**. That way, we are asking for a prediction before its simulated end-of-life point.

In [ ]:
data = validate_measurements(pd.read_csv("sample_data/demo_capacitors.csv"))
cap = data[data["Cap_ID"] == "AEC_001"].head(15)
result = assess_capacitor(cap, model, metadata)

print("Capacitor:", result["cap_id"])
print("Current age:", result["hours_observed"], "hours")
print("Predicted RUL (DEMO ONLY):", round(result["rul_hours"]), "hours")
print("Status:", result["status"])
print("Out-of-training warnings:", result["warnings"])


## 4. Look at the changing health indicators


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, col in zip(axes.ravel(), ["C_ratio", "ESR_ratio", "Leakage_ratio", "Case_Strain_Index"]):
    ax.plot(cap["Hours"], cap[col], marker="o")
    ax.set(title=col, xlabel="Hours")
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 5. Start the browser-based web app
Colab is for experimenting with the AI engine. The web interface itself is `app.py` in the ZIP. For local use, see the README and run `streamlit run app.py` on your computer. For a public URL, upload the project contents to GitHub and deploy to **Streamlit Community Cloud**.

**Scientific caution:** The synthetic network is not validated on real accelerated-life-test results. Do not use it for maintenance decisions.